# A.10 對話很長，最新的條件如何驗收？


你請助理提醒攝影社活動，先說下午三點集合，過幾天改成四點，又把提醒的寫法改成兩個條列。現在問「請提醒我十月十日何時在哪集合」，應得到新時間、原地點與新格式。不能只記住第一次的安排，也不能因為最後一則訊息沒提地點，就把地點丟掉。

先讀同一段對話中的三則訊息。左欄是訊息發生的日期，活動日期一直是十月十日。

| 訊息日期 | 使用者說的話 |
| --- | --- |
| 10月1日 | 10月10日攝影社活動，15:00在活動室M17集合。提醒請用一小段文字。 |
| 10月3日 | 同一場活動改成16:00集合，地點不變。 |
| 10月4日 | 提醒改成兩個條列，每項只寫一句。 |

問目前安排時，可以先在紙上圈出每個欄位仍然有效的內容：時間是十月三日改過的 `16:00`，地點仍是 `M17`，格式是十月四日要求的兩個條列。「更新」在這裡指某個條件被明確替換；只改時間，就不連帶改掉活動日期與地點。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/rewrite-A-conversation-updates.svg")))

圖中的每個框整理當時有效的條件。現在依最新安排作答，期待的是：

- 10月10日16:00集合。
- 地點是活動室M17。

這份紙上答案讓內容與偏好都可分開核對。若答成 `15:00`，沿用了被改掉的時間；若時間、地點都對，卻寫成一大段，則沒有符合目前的提醒格式。兩種錯誤應各自留下來，不能用「記得活動」這個模糊標籤合在一起。

同一段對話還可以換一個時間範圍來問：「截至10月2日，我當時安排幾點集合？」這題的答案是 `15:00`。它問的是更新以前的安排，所以不能拿現在的 `16:00` 覆蓋過去。核對時要先看問題問「目前」還是「截至某日」，再選那個時間範圍內有效的條件。記住新值與能回答舊值，是兩個不同要求。

再問：「活動結束後去哪裡喫晚餐？」三則訊息都沒有提供晚餐安排。合理的回答例如「對話中還沒有晚餐地點；你希望去哪裡喫？」這次應指出缺少的資訊並澄清，而不是把活動室當餐廳，也不是隨手補上一個店名。有明確條件的題目要能回答，沒有條件的題目要能停下來補問，兩種都需要驗收；每題都說不知道也不能完成任務。

若之後加入很長的其他對話，仍可保留這三種問題：目前安排、指定日期的舊安排，以及從未提供的安排。每題保存模型實際收到的完整歷史、問題與回答，分別檢查時間、地點、格式和是否需要澄清。較短版本也保留同樣的三則關鍵訊息，讓我們先知道它是否能處理更新規則，再看增加距離和幹擾後是否仍穩定。

這裡先假定三則訊息全部送入模型。若另用摘要或檢索來保留歷史，則要沿 [A.5](https://birdhackor.github.io/tiny-perceptron-vlm/A.5.html) 的區分，先查新時間及格式是否真的找回，再查回答是否用對；找回舊訊息卻漏掉更新，是資料選取的一種失敗，不能全算成模型讀了全文仍記錯。反過來，全文已收到也不代表最新條件一定被正確使用。

<details>
<summary>補充：來源與例子的界線</summary>

這組有限示例受到 Wu et al., [*LongMemEval: Benchmarking Chat Assistants on Long-Term Interactive Memory*，arXiv:2410.10813v2](https://arxiv.org/pdf/2410.10813v2)，§3.1–3.4 啟發。原評測有500個問題，涵蓋資訊擷取、跨對話時段推理、知識更新、時間推理與資訊不存在時棄答。§3.4 也分開完整歷史閱讀與記憶／檢索系統等條件。本節使用自行寫出的三則訊息與明確紙上答案，沒有使用原資料集或原評分程序，也沒有執行模型評測，不能據此宣稱助理已具備一般長期記憶能力。

</details>
